<a href="https://colab.research.google.com/github/RonazAbbasi/ml-zoomcamp2026/blob/main/Homework3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

# 1. Load the official dataset
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

# 2. Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# 3. Set the target column
target = "converted"

if target not in df.columns:
    raise ValueError(
        f"Target column '{target}' not found. "
        f"Available columns: {df.columns.tolist()}"
    )

# Convert target to numeric if necessary
if not pd.api.types.is_numeric_dtype(df[target]):
    df[target] = df[target].astype(str).str.lower().map(
        {"yes": 1, "no": 0, "true": 1, "false": 0}
    )

if df[target].isnull().any():
    raise ValueError("The target column contains missing or unrecognized values.")

y = df[target].astype(int)
X = df.drop(columns=[target])

# 4. Mutual information for categorical features
categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nMutual information with target:")
mi_scores = {}

for col in categorical_features:
    values = X[col].fillna("MISSING").astype(str)
    mi_scores[col] = mutual_info_score(values, y)

mi_table = pd.Series(mi_scores).sort_values(ascending=False)
display(mi_table.to_frame("mutual_information"))

# 5. Correlation of numeric features with the target
numeric_features = X.select_dtypes(include=np.number).columns.tolist()

print("\nNumeric feature correlations with target:")
if numeric_features:
    correlations = X[numeric_features].corrwith(y).sort_values(
        key=lambda s: s.abs(), ascending=False
    )
    display(correlations.to_frame("correlation_with_target"))

# 6. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 7. Prepare data for DictVectorizer
X_train_dict = X_train.copy()
X_test_dict = X_test.copy()

for col in X_train_dict.columns:
    if (
        pd.api.types.is_object_dtype(X_train_dict[col])
        or isinstance(X_train_dict[col].dtype, pd.CategoricalDtype)
        or pd.api.types.is_bool_dtype(X_train_dict[col])
    ):
        X_train_dict[col] = X_train_dict[col].fillna("MISSING").astype(str)
        X_test_dict[col] = X_test_dict[col].fillna("MISSING").astype(str)
    else:
        median_value = X_train_dict[col].median()
        X_train_dict[col] = X_train_dict[col].fillna(median_value)
        X_test_dict[col] = X_test_dict[col].fillna(median_value)

# Convert rows into dictionaries
train_dicts = X_train_dict.to_dict(orient="records")
test_dicts = X_test_dict.to_dict(orient="records")

dv = DictVectorizer(sparse=True)
X_train_encoded = dv.fit_transform(train_dicts)
X_test_encoded = dv.transform(test_dicts)

# 8. Train logistic regression
model = LogisticRegression(max_iter=2000, random_state=42)
model.fit(X_train_encoded, y_train)

# 9. Evaluate model
y_pred = model.predict(X_test_encoded)
accuracy = accuracy_score(y_test, y_pred)

print("\nTrain rows:", len(X_train))
print("Test rows:", len(X_test))
print("Accuracy:", round(accuracy, 4))

# 10. Show the most influential model features
feature_names = dv.get_feature_names_out()
coefficients = model.coef_[0]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})
feature_importance["abs_coefficient"] = (
    feature_importance["coefficient"].abs()
)

print("\nFeatures with largest absolute coefficients:")
display(
    feature_importance
    .sort_values("abs_coefficient", ascending=False)
    .head(15)
    .drop(columns="abs_coefficient")
)

Dataset shape: (5000, 9)
Columns: ['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1



Missing values:
lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

Mutual information with target:


,mutual_information
lead_source,0.033064
employment_status,0.022797
industry,0.002280
location,0.001064



Numeric feature correlations with target:


,correlation_with_target
lead_score,0.490206
interaction_count,0.448624
number_of_courses_viewed,0.395038
annual_income,0.181752



Train rows: 4000
Test rows: 1000
Accuracy: 0.72

Features with largest absolute coefficients:


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,feature,coefficient
14,lead_score,2.967019
4,employment_status=student,-0.622182
18,lead_source=paid_ads,-0.451457
7,industry=education,-0.442915
26,location=south_america,-0.370888
22,location=africa,-0.361166
15,lead_source=MISSING,-0.361151
16,lead_source=events,-0.350221
9,industry=healthcare,-0.340910
5,employment_status=unemployed,-0.335134
